# IVAP for Regression - Experimental results

In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Dict, Optional
import pandas as pd


artificial_datasets = ['linear_gaussian', 'nonlinear_sine', 'heteroscedastic',
                       'heavy_tailed', 'outliers', 'sparse_highdim', 'covariate_shift']
friedman_datasets = ['friedman1', 'friedman2', 'friedman3']

uci_datasets = ['airfoil', 'climate_bias', 'electricity']
other_datasets = ['star']

dataset_dict = dict()
dataset_dict['synthetic_datasets'] = artificial_datasets + friedman_datasets
dataset_dict['real_datasets'] = uci_datasets + other_datasets

import warnings
warnings.filterwarnings("ignore")


## Read results

In [ ]:
datasets = 'synthetic_datasets'
noise_level = 3
n_samples = 10000
if datasets == 'synthetic_datasets':
    df_summary = pd.read_csv(
    'output/'+ datasets + '_noise_' +
                      str(int(noise_level)) + '_' + str(int(n_samples)) + '.csv').iloc[:, 1:]
else:
    df_summary = pd.read_csv('output/real_datasets.csv').iloc[:, 1:]

In [ ]:
df_summary.head()

## Summarise and export to Tex 

In [ ]:
scenarios=dataset_dict[datasets]
for sel in scenarios:
    df_summ = df_summary[df_summary.scenario==sel].copy()
    df_summ['calibration']='base'
    df_summ['base_model']=[i.split(' ')[0] for i in df_summ.iloc[:,1].values]
    df_summ.iloc[['CVAP' in row for row in df_summ['model'].values], -2] = [i.split('C')[1] for i in df_summ.iloc[:,1].values if len(i.split(' ')) > 1]
    print(sel)
    df_pivot = df_summ.pivot(index='base_model', columns='calibration', values='mean')
    average_values = df_pivot.select_dtypes(include=['number']).mean()
    df_pivot.loc['mean'] = average_values
    

In [ ]:
df_summ['base_model'].unique()

In [ ]:
df_summ = df_summary[df_summary.scenario==sel].copy()
df_summ['calibration']='base'
df_summ['base_model']=[i.split(' ')[0] for i in df_summ.iloc[:,1].values]

In [ ]:
scen_desc = {}
if datasets == 'synthetic_datasets':
    scen_desc[scenarios[0]] = 'linear Gaussian'
    scen_desc[scenarios[1]] = 'nonlinear'
    scen_desc[scenarios[2]] = 'heteroscedastic noise'
    scen_desc[scenarios[3]] = 'heavy-tailed noise'
    scen_desc[scenarios[4]] = 'outlier contamination'
    scen_desc[scenarios[5]] = 'sparse high-dimensional signals'
    scen_desc[scenarios[6]] = 'covariate shift'
    scen_desc[scenarios[7]] = 'Friedman 1'
    scen_desc[scenarios[8]] = 'Friedman 2'
    scen_desc[scenarios[9]] = 'Friedman 3'
else:
    for scen in real_datasets:
        scen_desc[scen] = scen


In [ ]:
scenario = scenarios[0]

for scenario in scenarios:
    df_summ = df_summary[df_summary.scenario==scenario].copy()
    df_summ['calibration']='base'
    df_summ['base_model']=[i.split(' ')[0] for i in df_summ.iloc[:,1].values]
    df_summ.iloc[['CVAP' in row for row in df_summ['model'].values], -2] = [i.split('C')[1] for i in df_summ.iloc[:,1].values if len(i.split(' ')) > 1]
    print(scenario)
    df_pivot = df_summ.pivot(index='base_model', columns='calibration', values='mean')
    df_pivot_std = df_summ.pivot(index='base_model', columns='calibration', values='std')
    average_values = df_pivot.select_dtypes(include=['number']).mean()
    df_pivot.loc['average'] = average_values
    df_pivot = df_pivot.iloc[:, [-1, 0, 1]]
    df_pivot.columns = ['none', 'CVAR1', 'CVAR10']
    df_pivot.index.name = 'base model'
    display(df_pivot.round(3))

    scratch = df_pivot.copy()
    df_s = scratch.style.format("{:.3f}")
    for row in scratch.index:
        col = scratch.loc[row].idxmin()
        df_s = df_s.format(lambda x: "\\textbf{" + f'{x:.3f}' + "}", subset=(row, col))

    tex_output = df_s.format_index(precision=0).to_latex(
        hrules=True, 
        column_format='l|l|rrr', 
        multicol_align='c',
        caption = 'RMSE  - ' + scen_desc[scenario] + ' dataset ')
    print(tex_output)


In [ ]:
# df_summary.to_csv('all_synthetic_noise_3.csv')